# Telco Churn — Modeling Walkthrough

Companion notebook to `01_telco_churn_analysis.ipynb`. The first notebook
asked *what drives churn?* — this one asks *can we predict it?*

The flow:

1. Load the cleaned dataset via the shared `clean_telco_churn()` helper.
2. Benchmark three candidates — Logistic Regression, Random Forest, Gradient Boosting — with 5-fold stratified CV.
3. Inspect ROC, PR, confusion matrix, feature importance, and threshold sweep for the winner.
4. Fit the winner on all data and save it so `app.py` (Streamlit) can load it.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from telco_churn.cleaning import clean_telco_churn
from telco_churn.model import benchmark, fit_final
from telco_churn.evaluate import (
    score_pipeline, classification_text, roc_points, pr_points,
    feature_importance, threshold_sweep,
)

raw = pd.read_csv(ROOT / "Telco-Customer-Churn.csv")
cleaned = clean_telco_churn(raw)
cleaned.head(3)

## 1. Benchmark three models

In [ ]:
results, fitted, (X_train, X_test, y_train, y_test) = benchmark(cleaned)
results

## 2. Pick the winner and inspect it

We pick the top row by CV AUC. Gradient Boosting usually wins here by a small
margin, but Logistic Regression is within a few points — worth remembering
since it's far easier to explain to stakeholders.

In [ ]:
winner_name = results.iloc[0]["model"]
winner = fitted[winner_name]
print(f"Winner: {winner_name}")

score_pipeline(winner, X_test, y_test)

In [ ]:
print(classification_text(winner, X_test, y_test))

## 3. ROC + PR curves

In [ ]:
roc = roc_points(winner, X_test, y_test)
pr = pr_points(winner, X_test, y_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(roc["fpr"], roc["tpr"])
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set(title="ROC", xlabel="False Positive Rate", ylabel="True Positive Rate")

axes[1].plot(pr["recall"], pr["precision"])
axes[1].set(title="Precision–Recall", xlabel="Recall", ylabel="Precision")
plt.tight_layout()

## 4. Feature importance

The one-hot encoder expands categorical columns, so feature names show up
prefixed with `cat__` or `num__`. Contract type, tenure, and total charges
consistently dominate.

In [ ]:
feature_importance(winner, top_n=15)

## 5. Threshold sweep

Default threshold is 0.5 — but the business cost of a false positive (handing
a retention discount to a customer who wouldn't have left) is very different
from a false negative (losing a customer entirely). Sweep to pick a threshold
that matches your operating point.

In [ ]:
threshold_sweep(winner, X_test, y_test)

## 6. Persist winner for the Streamlit app

In [ ]:
fit_final(cleaned, winner_name)
print("Saved. You can now run `streamlit run app.py` from the project root.")